In [3]:
import numpy as np
import pandas as pd
import folium
from branca.colormap import linear
from tqdm import tqdm

In [4]:
#clientes = pd.read_excel('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\dados.xlsx', sheet_name='zips')
clientes = pd.read_excel('/home/rcamargo/Dropbox/10-orientacoes-tcc/23-felipe-malta/TCC/dados.xlsx', sheet_name='zips')

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [6]:
df_clientes = clientes[['Order Postal Code', 'Latitude', 'Longitude']]

In [7]:
# Coordenadas da cidade de São Paulo
latitude_min = clientes['Latitude'].min()
latitude_max = clientes['Latitude'].max()
longitude_min = clientes['Longitude'].min()
longitude_max = clientes['Longitude'].max()

# Tamanho do quadrado em graus (aproximadamente 1 km)
tamanho_quadrado = 0.009

# Número de quadrados em latitude e longitude
num_quadrados_lat = int(np.ceil((latitude_max - latitude_min) / tamanho_quadrado))
num_quadrados_lon = int(np.ceil((longitude_max - longitude_min) / tamanho_quadrado))

# Criar o mapa centrado na cidade de São Paulo
mapa = folium.Map(location=[(latitude_min + latitude_max) / 2, (longitude_min + longitude_max) / 2], zoom_start=12)

df_quadrados = pd.DataFrame(columns=['id', 'maxLat', 'minLat', 'maxLong', 'minLong'])

# Iterar sobre os quadrados em latitude e longitude
i = 0
for lat in np.arange(latitude_min, latitude_min + num_quadrados_lat * tamanho_quadrado, tamanho_quadrado):
    for lon in np.arange(longitude_min, longitude_min + num_quadrados_lon * tamanho_quadrado, tamanho_quadrado):
            sw = [lat, lon]
            ne = [lat + tamanho_quadrado, lon + tamanho_quadrado]
            bounds = [sw, ne]
            folium.Rectangle(bounds, color='black', fill=False).add_to(mapa)
            df_quadrados.at[i, 'minLat'] = lat
            df_quadrados.at[i, 'minLong'] = lon
            df_quadrados.at[i, 'maxLat'] = lat + tamanho_quadrado
            df_quadrados.at[i, 'maxLong'] = lon + tamanho_quadrado
            df_quadrados.at[i, 'id'] = i
            i = i+1

# Exibir o mapa com os quadrados
mapa


In [8]:
# Função otimizada para alocar o cliente em um quadrado
def alocar_cliente(cliente, quadrados_dict):
    global pbar
    pbar.update(1)
    for _, quadrado in quadrados_dict.items():
        if (
            quadrado['minLat'] <= cliente['Latitude'] <= quadrado['maxLat'] and
            quadrado['minLong'] <= cliente['Longitude'] <= quadrado['maxLong']
        ):
            return quadrado['id']
    return None

# Converter o DataFrame 'quadrados' em um dicionário
quadrados_dict = {row['id']: row for _, row in df_quadrados.iterrows()}

# Adicionar coluna para alocar os clientes nos quadrados
with tqdm(total=len(df_clientes)) as pbar:
    df_clientes['Quadrado'] = df_clientes.apply(lambda row: alocar_cliente(row, quadrados_dict), axis=1)
    
df_result = df_clientes.merge(df_quadrados, left_on='Quadrado', right_on='id').drop('id', axis=1)

100%|████████████████████████████████████▉| 35472/35473 [09:17<00:00, 50.56it/s]/tmp/ipykernel_21925/2002405190.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clientes['Quadrado'] = df_clientes.apply(lambda row: alocar_cliente(row, quadrados_dict), axis=1)
100%|█████████████████████████████████████| 35473/35473 [09:17<00:00, 63.58it/s]


In [13]:
df_result.to_csv('/home/rcamargo/Dropbox/10-orientacoes-tcc/23-felipe-malta/TCC/df_clientes_quadrados.csv')

In [14]:
df_quadrados.to_csv('/home/rcamargo/Dropbox/10-orientacoes-tcc/23-felipe-malta/TCC/df_quadrados.csv')